#Loading the mmm model,meta data and simulation for a forward pass

In [35]:
# %%- Load simulation metadata and raw spend data

import json
import numpy as np
import pandas as pd
import arviz as az
from pathlib import Path

base_path = Path("/boot")

# Load fitted posterior
idata = az.from_netcdf(
    base_path / "/content/hierarchical_mmm_model_2 (1).nc"
)

# Load metadata JSON
with open(
    base_path / "/content/model_2_simulation_metadata.json",
    "r"
) as file:
    metadata = json.load(file)

# Load raw simulation CSV
simulation_df = pd.read_csv(
    base_path / "/content/model_2_simulation_data.csv"
)

print("Posterior loaded:", idata.groups())
print("Simulation rows:", len(simulation_df))
print("Simulation columns:", simulation_df.columns.tolist())

Posterior loaded: ['posterior', 'sample_stats', 'observed_data', 'constant_data']
Simulation rows: 420
Simulation columns: ['Show', 'Season', 'Week_Number', 'Network_TV_Spend', 'Cable_TV_Spend', 'Digital_Spend', 'Model_2_Split']


#Before reconstructing the media transformation and calculating marginal ROAS, this section inspects all variables stored inside the fitted ArviZ `InferenceData` object.


In [36]:
# %% Inspect all variables saved in the fitted model

for group_name in idata.groups():
    group = getattr(idata, group_name)

    print(f"\n--- {group_name} ---")

    for variable_name in group.data_vars:
        print(
            variable_name,
            group[variable_name].dims,
            group[variable_name].shape
        )


--- posterior ---
alpha ('chain', 'draw', 'channel') (4, 1000, 3)
beta_controls ('chain', 'draw', 'control') (4, 1000, 6)
beta_global ('chain', 'draw', 'channel') (4, 1000, 3)
beta_show ('chain', 'draw', 'show', 'channel') (4, 1000, 8, 3)
half_saturation ('chain', 'draw', 'channel') (4, 1000, 3)
intercept_global ('chain', 'draw') (4, 1000)
intercept_show ('chain', 'draw', 'show') (4, 1000, 8)
intercept_show_season ('chain', 'draw', 'show_season') (4, 1000, 28)
log_beta_global ('chain', 'draw', 'channel') (4, 1000, 3)
mu_all ('chain', 'draw', 'obs') (4, 1000, 420)
sd_log_beta_show ('chain', 'draw', 'channel') (4, 1000, 3)
sd_show ('chain', 'draw') (4, 1000)
sd_show_season ('chain', 'draw') (4, 1000)
sigma ('chain', 'draw') (4, 1000)
z_beta_show ('chain', 'draw', 'show', 'channel') (4, 1000, 8, 3)
z_show ('chain', 'draw', 'show') (4, 1000, 8)
z_show_season ('chain', 'draw', 'show_season') (4, 1000, 28)

--- sample_stats ---
acceptance_rate ('chain', 'draw') (4, 1000)
diverging ('chain',

#Extracting the parameters from the NetCDF

In [37]:
posterior = idata.posterior.stack(
    sample=("chain", "draw")
)

alpha_draws = (
    posterior["alpha"]
    .transpose("sample", "channel")
    .values
)

half_saturation_draws = (
    posterior["half_saturation"]
    .transpose("sample", "channel")
    .values
)

beta_show_draws = (
    posterior["beta_show"]
    .transpose("sample", "show", "channel")
    .values
)

lagged_spend = (
    idata.constant_data["lagged_spend"]
    .values
)

valid_lags = (
    idata.constant_data["valid_lags"]
    .values
)

lag_numbers = (
    idata.constant_data["lag_numbers"]
    .values
)

show_index = (
    idata.constant_data["show_index"]
    .values
    .astype(int)
)

In [38]:
print(metadata.keys())

dict_keys(['spend_scale_model_2', 'y_mean_model_2', 'y_sd_model_2', 'MAX_LAG', 'GROUPS', 'SPEND', 'CHANNELS', 'show_levels'])


In [39]:
y_sd_model_2 = float(metadata["y_sd_model_2"])

spend_scale_model_2 = np.asarray(
    metadata["spend_scale_model_2"],
    dtype=float
)

CHANNELS = metadata["CHANNELS"]
SPEND = metadata["SPEND"]
GROUPS = metadata["GROUPS"]
MAX_LAG = int(metadata["MAX_LAG"])

print("y_sd_model_2:", y_sd_model_2)
print("spend_scale_model_2:", spend_scale_model_2)
print("CHANNELS:", CHANNELS)
print("MAX_LAG:", MAX_LAG)

y_sd_model_2: 45454.29355949826
spend_scale_model_2: [22909.9822823  22779.35652595 21892.69944891]
CHANNELS: ['Network TV', 'Cable TV', 'Digital']
MAX_LAG: 8


In [40]:
# %% Baseline media forward pass

# Stack chain and draw into one posterior-sample dimension
posterior = idata.posterior.stack(
    sample=("chain", "draw")
)

alpha_draws = (
    posterior["alpha"]
    .transpose("sample", "channel")
    .values
)

half_saturation_draws = (
    posterior["half_saturation"]
    .transpose("sample", "channel")
    .values
)

beta_show_draws = (
    posterior["beta_show"]
    .transpose("sample", "show", "channel")
    .values
)

# Stored model inputs
lagged_spend = idata.constant_data["lagged_spend"].values
valid_lags = idata.constant_data["valid_lags"].values
lag_numbers = idata.constant_data["lag_numbers"].values

show_index = (
    idata.constant_data["show_index"]
    .values
    .astype(int)
)

# Geometric-adstock weights
# Shape: samples × channels × lags
adstock_weights = (
    alpha_draws[:, :, None]
    ** lag_numbers[None, None, :]
)

# Normalized geometric adstock
# Result shape: samples × observations × channels
adstock_numerator = np.einsum(
    "ocl,scl->soc",
    lagged_spend,
    adstock_weights
)

adstock_denominator = np.einsum(
    "ol,scl->soc",
    valid_lags,
    adstock_weights
)

adstocked_media = (
    adstock_numerator
    / np.maximum(adstock_denominator, 1e-12)
)

# Saturation: x / (x + half_saturation)
saturated_media = (
    adstocked_media
    / (
        adstocked_media
        + half_saturation_draws[:, None, :]
    )
)

# Match each observation to its show's media coefficients
beta_by_observation = beta_show_draws[
    :,
    show_index,
    :
]

# Contribution in actual revenue units
baseline_contribution = (
    saturated_media
    * beta_by_observation
    * y_sd_model_2
)

print("alpha:", alpha_draws.shape)
print("half saturation:", half_saturation_draws.shape)
print("beta show:", beta_show_draws.shape)
print("lagged spend:", lagged_spend.shape)
print("adstocked media:", adstocked_media.shape)
print("saturated media:", saturated_media.shape)
print("baseline contribution:", baseline_contribution.shape)

assert baseline_contribution.shape == (
    4000,
    420,
    3
)

print("\nBaseline forward pass completed.")

alpha: (4000, 3)
half saturation: (4000, 3)
beta show: (4000, 8, 3)
lagged spend: (420, 3, 9)
adstocked media: (4000, 420, 3)
saturated media: (4000, 420, 3)
baseline contribution: (4000, 420, 3)

Baseline forward pass completed.


In [41]:
print(simulation_df.columns.tolist())
print(simulation_df[SPEND].head())
print(simulation_df[SPEND].shape)

['Show', 'Season', 'Week_Number', 'Network_TV_Spend', 'Cable_TV_Spend', 'Digital_Spend', 'Model_2_Split']
   Network_TV_Spend  Cable_TV_Spend  Digital_Spend
0       4028.297704    12084.893113   10070.744261
1       4854.153125    14562.459375   12135.382813
2       5370.453364    16111.360093   13426.133411
3       7619.321690    22857.965070   19048.304225
4       7133.216268    21399.648804   17833.040670
(420, 3)


In [42]:
# %% Build lag tensor from raw scaled spend

def make_lag_tensor(
    scaled_spend,
    group_values,
    max_lag
):
    """
    Build lagged media-spend and valid-lag tensors.

    Parameters
    ----------
    scaled_spend : np.ndarray
        Shape: observations × channels

    group_values : array-like
        One grouping key per observation.
        For this model, each key represents the combination
        of Show and Season.

    max_lag : int
        Maximum number of lag periods.

    Returns
    -------
    lag_tensor : np.ndarray
        Shape: observations × channels × lags

    valid_tensor : np.ndarray
        Shape: observations × lags
    """

    scaled_spend = np.asarray(
        scaled_spend,
        dtype=float
    )

    group_values = np.asarray(
        group_values,
        dtype=object
    )

    n_obs, n_channels = scaled_spend.shape
    n_lags = max_lag + 1

    if len(group_values) != n_obs:
        raise ValueError(
            "group_values must contain one grouping key "
            "for every observation."
        )

    lag_tensor = np.zeros(
        (n_obs, n_channels, n_lags),
        dtype=float
    )

    valid_tensor = np.zeros(
        (n_obs, n_lags),
        dtype=float
    )

    for row in range(n_obs):
        for lag in range(n_lags):

            source_row = row - lag

            if source_row < 0:
                continue

            same_group = (
                group_values[source_row]
                == group_values[row]
            )

            if same_group:
                lag_tensor[row, :, lag] = (
                    scaled_spend[source_row, :]
                )

                valid_tensor[row, lag] = 1.0

    return lag_tensor, valid_tensor


# %% Prepare raw and scaled media spend

raw_spend = simulation_df[
    SPEND
].to_numpy(dtype=float)

scaled_spend = (
    raw_spend
    / spend_scale_model_2[None, :]
)

# GROUPS = ["Show", "Season"]
# Combine both columns into one tuple per row
group_values = np.empty(
    len(simulation_df),
    dtype=object
)

group_values[:] = list(
    simulation_df[GROUPS]
    .astype(str)
    .itertuples(
        index=False,
        name=None
    )
)

print("Raw spend shape:", raw_spend.shape)
print("Scaled spend shape:", scaled_spend.shape)
print("Number of grouping keys:", len(group_values))
print("First grouping keys:", group_values[:5])


# %% Reconstruct baseline lag tensors

baseline_lag_tensor, reconstructed_valid_lags = (
    make_lag_tensor(
        scaled_spend=scaled_spend,
        group_values=group_values,
        max_lag=MAX_LAG
    )
)

print(
    "\nReconstructed lag tensor shape:",
    baseline_lag_tensor.shape
)

print(
    "Reconstructed valid-lag shape:",
    reconstructed_valid_lags.shape
)


# %% Compare against tensors saved with the fitted model

lag_tensor_matches = np.allclose(
    baseline_lag_tensor,
    lagged_spend,
    rtol=1e-10,
    atol=1e-10
)

valid_lags_match = np.allclose(
    reconstructed_valid_lags,
    valid_lags,
    rtol=1e-10,
    atol=1e-10
)

maximum_lag_difference = np.max(
    np.abs(
        baseline_lag_tensor
        - lagged_spend
    )
)

maximum_valid_lag_difference = np.max(
    np.abs(
        reconstructed_valid_lags
        - valid_lags
    )
)

print(
    "\nLag tensor matches saved model:",
    lag_tensor_matches
)

print(
    "Valid lags match saved model:",
    valid_lags_match
)

print(
    "Maximum lag difference:",
    maximum_lag_difference
)

print(
    "Maximum valid-lag difference:",
    maximum_valid_lag_difference
)


# Stop before mROAS if reconstruction is inconsistent
if not lag_tensor_matches:
    raise ValueError(
        "The reconstructed lag tensor does not match "
        "the lag tensor stored in the fitted model."
    )

if not valid_lags_match:
    raise ValueError(
        "The reconstructed valid-lag tensor does not match "
        "the tensor stored in the fitted model."
    )

print(
    "\nLag reconstruction completed successfully."
)

Raw spend shape: (420, 3)
Scaled spend shape: (420, 3)
Number of grouping keys: 420
First grouping keys: [('Show1', '1') ('Show1', '1') ('Show1', '1') ('Show1', '1')
 ('Show1', '1')]

Reconstructed lag tensor shape: (420, 3, 9)
Reconstructed valid-lag shape: (420, 9)

Lag tensor matches saved model: True
Valid lags match saved model: True
Maximum lag difference: 0.0
Maximum valid-lag difference: 0.0

Lag reconstruction completed successfully.


In [43]:
# %% Posterior media forward-pass function

def media_forward_pass(
    lag_tensor,
    valid_tensor
):
    """
    Calculate posterior channel contributions.

    Returns
    -------
    contribution : np.ndarray
        Shape: posterior samples × observations × channels
    """

    numerator = np.einsum(
        "ocl,scl->soc",
        lag_tensor,
        adstock_weights
    )

    denominator = np.einsum(
        "ol,scl->soc",
        valid_tensor,
        adstock_weights
    )

    adstocked = (
        numerator
        / np.maximum(denominator, 1e-12)
    )

    saturated = (
        adstocked
        / (
            adstocked
            + half_saturation_draws[:, None, :]
        )
    )

    contribution = (
        saturated
        * beta_by_observation
        * y_sd_model_2
    )

    return contribution


# %% Baseline posterior contributions

baseline_contribution = media_forward_pass(
    baseline_lag_tensor,
    reconstructed_valid_lags
)

print(
    "Baseline contribution shape:",
    baseline_contribution.shape
)


# %% Calculate posterior mROAS

perturbation_pct = 0.01

mroas_draws = {}
summary_rows = []

for channel_idx, channel_name in enumerate(CHANNELS):

    # Copy original spend
    perturbed_raw_spend = raw_spend.copy()

    # Increase one channel by 1% across all rows
    perturbed_raw_spend[:, channel_idx] *= (
        1 + perturbation_pct
    )

    # Apply original spend scaling
    perturbed_scaled_spend = (
        perturbed_raw_spend
        / spend_scale_model_2[None, :]
    )

    # Rebuild lag tensors
    perturbed_lag_tensor, perturbed_valid_lags = (
        make_lag_tensor(
            scaled_spend=perturbed_scaled_spend,
            group_values=group_values,
            max_lag=MAX_LAG
        )
    )

    # Forward pass under perturbed spend
    perturbed_contribution = media_forward_pass(
        perturbed_lag_tensor,
        perturbed_valid_lags
    )

    # Incremental revenue for the perturbed channel
    incremental_revenue_draws = (
        perturbed_contribution[
            :,
            :,
            channel_idx
        ]
        - baseline_contribution[
            :,
            :,
            channel_idx
        ]
    ).sum(axis=1)

    # Actual additional spend
    incremental_spend = (
        perturbed_raw_spend[:, channel_idx].sum()
        - raw_spend[:, channel_idx].sum()
    )

    # Posterior mROAS distribution
    channel_mroas = (
        incremental_revenue_draws
        / incremental_spend
    )

    mroas_draws[channel_name] = channel_mroas

    summary_rows.append({
        "Channel": channel_name,
        "Median mROAS": np.median(channel_mroas),
        "Mean mROAS": np.mean(channel_mroas),
        "P05": np.quantile(channel_mroas, 0.05),
        "P95": np.quantile(channel_mroas, 0.95),
        "Probability mROAS > 1": np.mean(
            channel_mroas > 1
        ),
        "Incremental Spend": incremental_spend
    })


# %% Display mROAS summary

mroas_summary = pd.DataFrame(summary_rows)

display(
    mroas_summary.round(3)
)

Baseline contribution shape: (4000, 420, 3)


,Channel,Median mROAS,Mean mROAS,P05,P95,Probability mROAS > 1,Incremental Spend
0,Network TV,2.438,2.371,1.520,2.977,1.000,45700.060
1,Cable TV,1.901,1.924,1.523,2.403,1.000,45251.352
2,Digital,1.158,1.196,0.842,1.670,0.778,49237.680


###Interpretation

###Network TV provides the strongest marginal return. A small increase in Network TV spending is estimated to generate a median of ₹2.44 in additional revenue for every extra ₹1 spent. The 90% credible range is ₹1.52 to ₹2.98, and all posterior estimates are above 1. This gives strong evidence that additional Network TV investment is likely to generate revenue above its incremental media cost.

###Cable TV is the second-best channel. Every additional ₹1 spent is estimated to generate a median of ₹1.90 in incremental revenue, with a credible range of ₹1.52 to ₹2.40. Its probability of mROAS exceeding 1 is also 100%, indicating strong support for increasing spend, although its expected marginal return is lower than Network TV.

###Digital has a positive but less certain marginal return. Every additional ₹1 spent is estimated to generate a median of ₹1.16 in incremental revenue, with a credible range of ₹0.84 to ₹1.67. Since the lower bound falls below 1 and the probability of mROAS exceeding 1 is 77.8%, Digital may still produce a positive return, but there is meaningful uncertainty that additional spending could fail to recover its media cost.

###Business recommendation

The model suggests prioritizing incremental budget in this order:

Network TV → Cable TV → Digital

Network TV offers the highest expected marginal return and strong certainty. Cable TV also appears attractive and reliable. Digital should be scaled more cautiously, potentially through smaller increases or controlled experiments.

The incremental-spend figures—approximately ₹45,700 for Network TV, ₹45,251 for Cable TV, and ₹49,238 for Digital—represent the 1% spending increases used in the simulation. These mROAS estimates apply around the current spending levels and should not be assumed to remain constant for large budget increases because returns may decline as the channels become more saturated.

In [44]:
# %% Show-wise mROAS

show_column = "Show"
perturbation_pct = 0.01

show_mroas_draws = {}
show_summary_rows = []

for show_name in simulation_df[show_column].unique():

    show_mask = (
        simulation_df[show_column]
        .astype(str)
        .to_numpy()
        == str(show_name)
    )

    for channel_idx, channel_name in enumerate(CHANNELS):

        perturbed_raw_spend = raw_spend.copy()

        # Increase spend only for this show and this channel
        perturbed_raw_spend[
            show_mask,
            channel_idx
        ] *= (1 + perturbation_pct)

        perturbed_scaled_spend = (
            perturbed_raw_spend
            / spend_scale_model_2[None, :]
        )

        perturbed_lag_tensor, perturbed_valid_lags = (
            make_lag_tensor(
                scaled_spend=perturbed_scaled_spend,
                group_values=group_values,
                max_lag=MAX_LAG
            )
        )

        perturbed_contribution = media_forward_pass(
            perturbed_lag_tensor,
            perturbed_valid_lags
        )

        # Revenue impact only within the selected show
        incremental_revenue_draws = (
            perturbed_contribution[
                :,
                show_mask,
                channel_idx
            ]
            - baseline_contribution[
                :,
                show_mask,
                channel_idx
            ]
        ).sum(axis=1)

        incremental_spend = (
            perturbed_raw_spend[
                show_mask,
                channel_idx
            ].sum()
            - raw_spend[
                show_mask,
                channel_idx
            ].sum()
        )

        if incremental_spend <= 0:
            continue

        channel_mroas = (
            incremental_revenue_draws
            / incremental_spend
        )

        key = (show_name, channel_name)
        show_mroas_draws[key] = channel_mroas

        show_summary_rows.append({
            "Show": show_name,
            "Channel": channel_name,
            "Median mROAS": np.median(channel_mroas),
            "Mean mROAS": np.mean(channel_mroas),
            "P05": np.quantile(channel_mroas, 0.05),
            "P95": np.quantile(channel_mroas, 0.95),
            "Probability mROAS > 1": np.mean(
                channel_mroas > 1
            ),
            "Incremental Spend": incremental_spend
        })

show_mroas_summary = pd.DataFrame(
    show_summary_rows
)

show_mroas_summary = (
    show_mroas_summary
    .sort_values(
        ["Show", "Median mROAS"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(
    show_mroas_summary.round(3)
)

,Show,Channel,Median mROAS,Mean mROAS,P05,P95,Probability mROAS > 1,Incremental Spend
0,Show1,Cable TV,2.759,2.761,2.338,3.177,1.000,9480.189
1,Show1,Network TV,2.635,2.652,2.253,3.103,1.000,10527.252
2,Show1,Digital,1.551,1.542,1.099,1.961,0.971,10949.369
3,Show2,Cable TV,1.496,1.509,0.970,2.076,0.942,5302.069
4,Show2,Network TV,1.168,1.148,0.493,1.695,0.688,7217.651
5,Show2,Digital,0.795,0.818,0.420,1.295,0.217,7834.180
6,Show3,Network TV,3.987,3.728,1.074,5.339,0.957,10692.326
7,Show3,Cable TV,1.791,1.967,1.185,3.438,0.986,10655.006
8,Show3,Digital,0.776,0.941,0.261,2.295,0.333,10236.228
9,Show4,Cable TV,1.461,1.477,0.985,2.020,0.946,7219.451


#Showwise MROAS

In [45]:
show_mroas_pivot = (
    show_mroas_summary
    .pivot(
        index="Show",
        columns="Channel",
        values="Median mROAS"
    )
)

display(
    show_mroas_pivot.round(3)
)

Channel,Cable TV,Digital,Network TV
Show,,,
Show1,2.759,1.551,2.635
Show2,1.496,0.795,1.168
Show3,1.791,0.776,3.987
Show4,1.461,1.184,1.171
Show5,1.422,0.952,0.757
Show6,1.697,1.140,3.450
Show7,1.359,0.979,1.674
Show8,1.598,1.950,1.251


###Based on the hierarchical Bayesian MMM, I would recommend shifting incremental budget first toward Network TV, then Cable TV, while treating Digital as a selective/test-and-learn channel. Network TV has the strongest overall marginal return, Cable TV is consistently reliable, and Digital should be increased only for shows where its show-level mROAS is clearly attractive, such as Show8. For larger budget changes, I would rerun scenario simulations and keep spend within historical support because marginal returns can decline due to saturation.